[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/00_question_orientation.ipynb)

# R1-Q2 Week 1 — Question orientation and pre-commit

This notebook does WGCNA-specific preparation for R1-Q2 and produces the consolidated pre-commit file you submit at the end of Week 1.

By the end of this notebook you will have:

- Filtered shoot and root expression matrices ready for WGCNA in Week 2 (`shoot_filtered.parquet`, `root_filtered.parquet`).
- A written pre-commit covering five decision areas: network-construction choices, hub-identification choices, the known-regulators reference set, the statistical framework for the hub-vs-known comparison, and the background gene set.
- A consolidated `precommit.json` written to your output directory, ready as the Week 1 EQ#1 deliverable.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

#!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load and inspect

In this section you'll load the AtGenExpress expression data and its
sample metadata, combine the per-stress tables into a single expression
matrix, and split that matrix into a shoot table and a root table. By
the end of this section you'll have two per-tissue matrices ready for
the WGCNA-specific filtering step in Section 3.

The rationale-level `r1_orientation.ipynb` walked through the
AtGenExpress structure (eight stresses + control, shoot and root tissue,
time course up to 24 hours) and the three caveats that apply across all
R1 questions. Those caveats matter just as much for R1-Q2 as they did
for R1-Q1; refresh on them in the orientation notebook if you haven't
worked through it recently.

In [ ]:
# Load the AtGenExpress expression data and the sample-level metadata.
# Both functions hit the cache on subsequent runs in the same session.
data = iri.load_atgenexpress()
metadata = iri.atgenexpress_metadata()

print(f"Loaded {len(data)} stress conditions: {sorted(data.keys())}")
print(f"Metadata: {metadata.shape[0]} samples × {metadata.shape[1]} columns\n")
metadata.head()

In the above cell `data` is a dictionary with one DataFrame per stress (probes by samples).
The per-stress tables share the same probe index but have different
samples in their columns, so we can concatenate them column-wise to get
a single 22,810-probe-by-248-sample expression matrix.

Once we have that combined matrix, the metadata's `tissue` column tells
us which samples are shoot and which are root. We index into the matrix
to produce a shoot-only table and a root-only table.

In [ ]:
import pandas as pd

# Combine the per-stress matrices into a single probes × samples table.
# All AtGenExpress samples were measured on the same Affymetrix ATH1
# array, so the probe set is identical across stresses. We use an inner
# join to be explicit about that — any probe missing from any stress
# would be dropped, but in practice none are missing.
full = pd.concat(data.values(), axis=1, join="inner")

# Use the metadata's tissue column to identify the GSMs in each tissue,
# then index the combined matrix.
shoot_gsms = metadata.index[metadata["tissue"] == "shoot"]
root_gsms = metadata.index[metadata["tissue"] == "root"]
shoot = full[shoot_gsms]
root = full[root_gsms]

print(f"Combined:  {full.shape[0]:>5} probes × {full.shape[1]:>3} samples")
print(f"Shoot:     {shoot.shape[0]:>5} probes × {shoot.shape[1]:>3} samples")
print(f"Root:      {root.shape[0]:>5} probes × {root.shape[1]:>3} samples")

The shapes above carry a methodological warning worth absorbing before
Section 3. For a co-expression network, **samples are observations**
and **genes are features**. Each per-tissue table has on the order of
a hundred-plus samples for estimating relationships across 22,810
genes. That ratio — far more features than observations — is the same
setting that shaped Rationale 1's approach in `r1_orientation.ipynb`
and R1-Q1. WGCNA handles it through soft thresholding and module
detection, but the underlying constraint is unchanged: modeling
choices can fit noise as easily as signal. The pre-commits in
Sections 4–6 are how you keep yourself honest about that.

A three-digit sample count sounds comfortable, but it isn't — not when
the feature count is nearly two orders of magnitude larger.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

ax_shoot.hist(shoot.values.flatten(), bins=80, color="steelblue", log=True)
ax_shoot.set_title(f"Shoot expression values (n={shoot.shape[1]} samples)")
ax_shoot.set_xlabel("MAS5-normalized intensity")
ax_shoot.set_ylabel("Probe-sample count (log scale)")

ax_root.hist(root.values.flatten(), bins=80, color="seagreen", log=True)
ax_root.set_title(f"Root expression values (n={root.shape[1]} samples)")
ax_root.set_xlabel("MAS5-normalized intensity")

plt.tight_layout()
plt.show()

print("\nValue statistics:")
print(f"  shoot:  min={shoot.values.min():>7.1f}   median={np.median(shoot.values):>6.1f}   max={shoot.values.max():>7.1f}")
print(f"  root:   min={root.values.min():>7.1f}   median={np.median(root.values):>6.1f}   max={root.values.max():>7.1f}")

The distributions are right-skewed with a long tail — characteristic of
linear-scale MAS5-normalized microarray intensities. Before filtering on
variance in Section 3, you'll log-transform these values so that
high-expression genes don't dominate the variance calculation on
absolute-magnitude grounds alone. The data shape you've just seen is
the input; Section 3 turns it into the input WGCNA actually expects.

## 3) Filter low-variance genes

WGCNA does not need every gene on the array. Most of the 22,810 ATH1
probes measure genes that don't change appreciably across the
experimental conditions — they add noise to module detection without
carrying signal. Filtering to the genes that actually vary across
conditions makes the network smaller and the modules cleaner.

The standard "varies across conditions" measure in WGCNA-style work is
**MAD** (Median Absolute Deviation) — the robust analog of standard
deviation, less sensitive to outliers, and what Hakkak 2026 used to go
from ~21,000 genes to 5,251 in their conserved co-expression analysis.

One thing to do before applying MAD: **log-transform the expression
values.** On linear-scale data, a gene with mean expression 5,000 has
much larger absolute variation than a gene with mean expression 50,
even if both respond identically in fold-change terms. MAD on
linear-scale data would surface highly expressed housekeeping genes
regardless of whether they respond to stress; MAD on log-scale data
surfaces genes that change in fold-change terms, which is what we
actually care about.

You'll apply the filter independently to shoot and root — the
Considerations section commits to per-tissue networks, and a pooled
filter would walk that back. Each tissue gets its own MAD distribution
and its own cutoff.

In [ ]:
### 3.1) Log-transform the per-tissue tables
import numpy as np

# log2(x + 1) — the +1 keeps zero-valued entries from producing -inf.
shoot_log = np.log2(shoot + 1)
root_log = np.log2(root + 1)

print("Before log transform (linear-scale MAS5 intensities):")
print(f"  shoot:  min={shoot.values.min():>7.1f}   median={np.median(shoot.values):>6.1f}   max={shoot.values.max():>7.1f}")
print(f"  root:   min={root.values.min():>7.1f}   median={np.median(root.values):>6.1f}   max={root.values.max():>7.1f}")

print("\nAfter log2(x + 1) transform:")
print(f"  shoot:  min={shoot_log.values.min():>7.2f}   median={np.median(shoot_log.values):>6.2f}   max={shoot_log.values.max():>7.2f}")
print(f"  root:   min={root_log.values.min():>7.2f}   median={np.median(root_log.values):>6.2f}   max={root_log.values.max():>7.2f}")

### 3.1) Compute per-gene MAD

For each gene, **MAD** across samples is the **Median** of the **Absolute**
**Deviations** from the gene's median expression:

> MAD(gene) = median( |x_sample − median(x_samples)| )

A high-MAD gene varies a lot across conditions; a low-MAD gene stays
roughly constant. Compute this for every gene in each tissue:

In [ ]:
### 3.2) Calculate the MAD per gene across samples, and identify the most variable genes in each tissue.
def per_gene_mad(matrix):
    """MAD per gene across samples."""
    median = matrix.median(axis=1)
    return (matrix.sub(median, axis=0)).abs().median(axis=1)

shoot_mad = per_gene_mad(shoot_log)
root_mad = per_gene_mad(root_log)

print(f"Shoot MAD series: {len(shoot_mad)} genes")
print(f"Root MAD series:  {len(root_mad)} genes\n")

print("Top 5 genes by MAD in shoot:")
print(shoot_mad.sort_values(ascending=False).head())
print("\nTop 5 genes by MAD in root:")
print(root_mad.sort_values(ascending=False).head())

In [ ]:
### 3.3) Visualize the MAD distributions and identify a cutoff for the most variable genes.
fig, (ax_shoot, ax_root) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

# Cutoff value: top 25% by MAD per tissue. Compute the threshold and
# overlay a vertical line on each panel.
MAD_PERCENTILE = 75
shoot_mad_threshold = np.percentile(shoot_mad, MAD_PERCENTILE)
root_mad_threshold = np.percentile(root_mad, MAD_PERCENTILE)

ax_shoot.hist(shoot_mad, bins=80, color="steelblue")
ax_shoot.axvline(shoot_mad_threshold, color="black", linestyle="--", linewidth=1.5,
                 label=f"{MAD_PERCENTILE}th pctile = {shoot_mad_threshold:.2f}")
ax_shoot.set_title(f"Shoot per-gene MAD (log2 scale, n={len(shoot_mad)} genes)")
ax_shoot.set_xlabel("MAD (log2 units)")
ax_shoot.set_ylabel("Gene count")
ax_shoot.legend()

ax_root.hist(root_mad, bins=80, color="seagreen")
ax_root.axvline(root_mad_threshold, color="black", linestyle="--", linewidth=1.5,
                label=f"{MAD_PERCENTILE}th pctile = {root_mad_threshold:.2f}")
ax_root.set_title(f"Root per-gene MAD (log2 scale, n={len(root_mad)} genes)")
ax_root.set_xlabel("MAD (log2 units)")
ax_root.legend()

plt.tight_layout()
plt.show()

print(f"\nShoot {MAD_PERCENTILE}th-percentile MAD threshold: {shoot_mad_threshold:.3f}")
print(f"Root  {MAD_PERCENTILE}th-percentile MAD threshold: {root_mad_threshold:.3f}")

The MAD distribution is right-skewed in both tissues: most genes have
small MAD values (they don't change much across conditions), and a
smaller subset has substantial MAD values (the genes that drive
variation in the data). The 75th-percentile cutoff keeps the top 25%
of genes by MAD — roughly 5,700 genes per tissue, the same order of
magnitude as Hakkak 2026's 5,251.

A few words on the working default:

- **Top 25% by percentile, not top-5,251 by rank.** Hakkak's specific
  number was a consequence of their pipeline (13 datasets integrated
  with ComBat, post-correction probe count of ~21,000, MAD-filtered to
  5,251). It is not a WGCNA-canonical threshold. A percentile-based
  cutoff is defensible on its own terms and adapts if the gene count
  ever changes.

- **Independent per-tissue.** Shoot and root each get their own MAD
  distribution and their own cutoff. The two tissues will probably keep
  overlapping but distinct gene sets — which is what we want, since the
  whole point of building per-tissue networks is letting tissue-specific
  biology drive the result.

- **75 is a working default, not a fixed answer.** If you have a
  principled reason to choose a different percentile (for example, a
  scale-free-fit diagnostic in Notebook 1 suggests too many or too few
  genes), record the reasoning in EQ#1 and revise. The pre-commit you
  record in Section 7 captures both the value and the reasoning.

In [ ]:
### 3.4) Filter the log-transformed tables to keep only the most variable genes per tissue. 
# Keep genes above the 75th-percentile MAD threshold per tissue.
shoot_kept_genes = shoot_mad[shoot_mad >= shoot_mad_threshold].index
root_kept_genes = root_mad[root_mad >= root_mad_threshold].index

shoot_filtered = shoot_log.loc[shoot_kept_genes]
root_filtered = root_log.loc[root_kept_genes]

print(f"After MAD filter (top {100 - MAD_PERCENTILE}% per tissue):")
print(f"  shoot:  {shoot_filtered.shape[0]:>5} genes × {shoot_filtered.shape[1]:>3} samples")
print(f"  root:   {root_filtered.shape[0]:>5} genes × {root_filtered.shape[1]:>3} samples")

# Inspect overlap between the two filtered gene sets — useful reality check.
both = set(shoot_kept_genes) & set(root_kept_genes)
print(f"\nGenes in both filtered tables: {len(both)}")
print(f"  shoot-only: {len(set(shoot_kept_genes) - both)}")
print(f"  root-only:  {len(set(root_kept_genes) - both)}")

In [ ]:
# Save the log-transformed, MAD-filtered matrices for Notebook 1.
shoot_path = OUTPUT_DIR / "shoot_filtered.parquet"
root_path = OUTPUT_DIR / "root_filtered.parquet"
shoot_filtered.to_parquet(shoot_path)
root_filtered.to_parquet(root_path)

# Round-trip read-back to confirm the saves are intact.
assert pd.read_parquet(shoot_path).shape == shoot_filtered.shape
assert pd.read_parquet(root_path).shape == root_filtered.shape

print(f"Saved: {shoot_path}")
print(f"   ({shoot_filtered.shape[0]} genes × {shoot_filtered.shape[1]} samples)")
print(f"Saved: {root_path}")
print(f"   ({root_filtered.shape[0]} genes × {root_filtered.shape[1]} samples)")

The filtered matrices are saved and ready for Notebook 1's WGCNA
pipeline. The next section of *this* notebook is the first pre-commit
category — your network-construction choices for PyWGCNA.

A bookkeeping note before moving on: the gene-filtering decision you
just made (log-transform, top 25% by MAD, independent per tissue) is
itself a pre-commit decision. It needs to be recorded alongside the
others in `precommit.json` at Section 7. The opening-cells design notes
sketched a five-category schema (network construction, hub
identification, reference set, statistical framework, background); the
working plan is to add gene filtering as a sixth top-level category.
We'll settle the schema concretely in Section 4 — before any pre-commit
cells get drafted — so all six categories serialize uniformly.

## 4) Pre-commit your network-construction choices

This section is the first of three that build a single `precommit.json`
file — the Week 1 deliverable for EQ#1. Sections 4, 5, and 6 each record
one or two pre-commit categories; the gene-filtering decision you
already applied in Section 3 gets recorded alongside them. Six
categories total. Section 7 writes the combined `precommit` dictionary
to disk.

Every category in `precommit` records four fields:

- **`method`** — one-line name of the approach.
- **`params`** — the values you commit to in writing, before running.
- **`applied`** — values computed from data after applying the
  pre-commit (only for categories where there are any).
- **`reasoning`** — a sentence or two explaining the choice. This is
  what EQ#1 expands on.

Pre-committing matters because every choice in this kind of work — what
to filter, how to build a network, how to identify hubs, what to
compare against — could be tuned post-hoc to make the result look
better. The pre-commit is a written record that you made these
decisions before seeing the answer. PyWGCNA's defaults for
stress-response work are already sensible, so in most cases you'll
adopt them as your pre-commit. The act of writing them down is what
makes them yours rather than the tool's.

### 4.1) Network construction — what you're committing to

PyWGCNA needs five parameter choices before it can build a network.
Each one shapes either how genes are connected to each other or how
modules are detected from those connections.

- **`networkType = "signed hybrid"`** — When two genes have correlated
  expression, are positively-correlated and negatively-correlated genes
  treated the same (unsigned), or differently (signed/signed hybrid)?
  For stress-response work, you want them treated differently — a gene
  that goes *up* under stress and a gene that goes *down* under stress
  are doing different biology, even if their absolute correlation is
  the same. Signed hybrid is the WGCNA authors' recommendation for
  stress-response work and PyWGCNA's default.

- **`TOMType = "signed"`** — The Topological Overlap Matrix (TOM) is
  the secondary similarity measure WGCNA uses after raw correlation.
  Setting it to signed (rather than unsigned) keeps the sign
  distinction consistent with `networkType`. There is no good reason
  to mix a signed network with an unsigned TOM.

- **`RsquaredCut = 0.9`** — Soft-thresholding power selection looks
  for the lowest power at which the scale-free topology fit (R²)
  exceeds this threshold. 0.9 is the WGCNA-conventional cutoff. (You
  will actually run the power selection in Notebook 1; this is the
  threshold value the selection uses.)

- **`minModuleSize = 50`** — The smallest module size that survives
  the merging step. Smaller numbers (10–30) produce more, finer
  modules; larger numbers (100+) produce fewer, broader modules. 50 is
  PyWGCNA's default and a reasonable middle ground for a
  few-thousand-gene network.

- **`MEDissThres = 0.2`** — The module-eigengene dissimilarity
  threshold for merging similar modules. Two modules whose eigengenes
  are more than 1 − 0.2 = 0.8 correlated get merged. 0.2 is the
  PyWGCNA default; lower values keep more distinct modules.

The five values above are PyWGCNA's defaults for stress-response
co-expression analysis. Adopting them as your pre-commit is the
recommended path; if you have a principled reason to change any of
them, record that reasoning in the `reasoning` field of the cell below.

In [ ]:
network_construction = {
    "method": "WGCNA via PyWGCNA, signed hybrid network",
    "params": {
        "networkType": "signed hybrid",
        "TOMType": "signed",
        "RsquaredCut": 0.9,
        "minModuleSize": 50,
        "MEDissThres": 0.2,
    },
    "reasoning": (
        "Signed hybrid network keeps up-regulated and down-regulated "
        "stress responses distinguishable. PyWGCNA defaults are the "
        "WGCNA authors' recommendations for stress-response work; "
        "soft-thresholding power gets selected from data in Notebook 1 "
        "using the conventional R^2 >= 0.9 cutoff."
    ),
}

# No 'applied' field — soft-thresholding power is selected from data
# in Notebook 1's pickSoftThreshold step, not here.

print("Network construction pre-commit recorded:")
for key, value in network_construction["params"].items():
    print(f"  {key:>20} = {value!r}")

### 4.2) Build the `precommit` dictionary

Combine the gene-filtering decision from Section 3 with the
network-construction pre-commit you just made. Sections 5 and 6 add
the remaining four categories; Section 7 writes the assembled
dictionary to disk.

In [ ]:
# Gene filtering — recorded from the variables produced in Section 3.
# Cast numpy scalars to Python types so the dict is JSON-safe.
gene_filtering = {
    "method": "MAD top percentile per tissue, log2(x+1) transformed",
    "params": {
        "log_transform": "log2(x+1)",
        "percentile_cutoff": int(MAD_PERCENTILE),
        "independent_per_tissue": True,
    },
    "applied": {
        "shoot": {
            "mad_threshold": float(shoot_mad_threshold),
            "n_genes_kept": int(shoot_filtered.shape[0]),
        },
        "root": {
            "mad_threshold": float(root_mad_threshold),
            "n_genes_kept": int(root_filtered.shape[0]),
        },
    },
    "reasoning": (
        "Hakkak 2026 method (MAD on log-scale data). Top 25% by "
        "percentile rather than top-5,251 by rank — percentile is "
        "defensible on its own terms and adapts if the gene count "
        "changes. Independent per tissue per the Considerations "
        "commitment to per-tissue networks."
    ),
}

# Initialize the combined precommit dict.
precommit = {
    "gene_filtering": gene_filtering,
    "network_construction": network_construction,
}

print(f"`precommit` has {len(precommit)} of 6 categories filled in:")
for key in precommit:
    print(f"  - {key}")

## 5) Pre-commit your hub-identification choices

This section is the second of three building `precommit.json`. The hub-identification category covers three sub-decisions that have to be made in order, because each one depends on the previous:

1. **Stress-relevant module definition** — which modules count as "stress-relevant" and therefore worth searching for hubs in.
2. **Hub-ness metric** — once you have a set of stress-relevant modules, how do you score genes within them for hub-like behavior.
3. **Hub threshold** — once you have a score, where do you cut.

These three are grouped into one pre-commit category because they share a single sequenced decision chain. You can't pick hubs without first picking the stress-relevant modules; you can't apply a hub threshold without first picking the hub-ness metric.

The next three subsections walk through each decision in order.

### 5.1) Stress-relevant module definition

In a WGCNA network, modules are gene groupings detected from co-expression structure alone — the algorithm doesn't know which modules are "about stress." You need to add that knowledge from outside the network. Two methodologically defensible ways to do it:

- **Module–trait correlation.** For each module, compute the correlation between its module eigengene (the first principal component of its gene expression, summarizing the module's overall expression pattern) and a binary stress-vs-control label across samples. Modules whose eigengenes correlate strongly with the stress label are stress-relevant. Threshold convention: `|r| >= 0.3`, the value Langfelder & Horvath's WGCNA tutorial uses for module–trait significance work.

- **GO enrichment.** Run GO term enrichment on each module's gene list. Modules enriched for stress-related GO terms (e.g., "response to abiotic stimulus") are stress-relevant.

**Working default: module–trait correlation, `|r| >= 0.3`.** Two reasons. First, module–trait correlation is data-driven from the same dataset you're analyzing, with no dependence on the completeness of external annotation. Second, GO enrichment has a circular flavor for this question: if you select modules *for* stress-relevance via GO terms, finding that those modules contain stress regulators is partly built in. Module–trait correlation keeps the comparison cleaner.

The threshold value follows the WGCNA-tutorial convention. Hakkak 2026 used `|r| >= 0.5` for their consensus drought-and-salt analysis, but that was a two-condition contrast across multiple integrated datasets with stronger per-condition signal. AtGenExpress is a single-dataset nine-condition design where each per-stress correlation is computed against ~14 samples on average; a tighter threshold on the noisier per-stress signal would leave many genuinely stress-responsive modules on the table.

If you have a principled reason to use GO enrichment (or both), record the reasoning in the cell below.

### 5.2) Hub-ness metric: kME

The hub-ness metric is locked from the question page Considerations: **kME**, the correlation between a gene's expression and its module's eigengene. The four-way choice among kME, degree centrality, betweenness centrality, and eigenvector centrality was settled on the question page; this section records that decision in `precommit.json`.

Four reasons kME is the right choice:

1. **WGCNA-canonical.** WGCNA was designed around the module eigengene as the central abstraction; kME is the metric that measures how well a gene tracks that abstraction. The most internally consistent hub measure for this kind of analysis.

2. **Biologically interpretable.** A gene with kME = 0.9 in a module is one whose expression pattern is 90% correlated with the module's summary. Directly readable as biology. Betweenness and eigenvector centrality are graph-topological statements that need translation.

3. **PyWGCNA-native.** Computed by the package's standard pipeline; no additional library or implementation work needed.

4. **Empirically validated for this use case.** Hub genes identified by kME tend to be functionally central — over-represented for known stress regulators in published WGCNA studies of *Arabidopsis* stress response. This is the empirical observation that motivates R1-Q2's hypothesis in the first place.

No working-default discussion here because there's no real alternative on the table for this question.

### 5.3) Hub threshold

Once you have kME for every gene in every stress-relevant module, you need a threshold to call hubs. Two options:

- **Top-N per module.** Take the highest-kME N genes in each module, regardless of the absolute kME value. Guarantees every module contributes exactly N hubs. A common choice is N = 30.

- **Fixed kME cutoff.** Take every gene whose kME exceeds a fixed threshold θ, regardless of module size. Modules with tighter internal structure contribute more hubs; modules with looser structure contribute fewer (or zero). A common choice is θ = 0.8.

**Working default: fixed kME cutoff at θ = 0.8.** Three reasons. First, fixed-cutoff is the WGCNA convention — Langfelder & Horvath's original work and most downstream literature define hubs by an absolute kME threshold in the 0.7–0.8 range, not by ranking within the module. Second, "hub" should mean "high centrality," not "high-ranked relative to a small pool"; a top-30 gene in a 60-gene module is in the top half of that module, while a top-30 gene in a 500-gene module is in the top 6%, and those are different kinds of hubs. Third, the rationale-level precedent cited on the question page — Kumar et al. (2023) identifying 117 central transcription factors in an *Arabidopsis* drought co-expression study — used kME-based hub identification on stress data of this kind.

A consequence to expect: some stress-relevant modules may yield no qualifying hubs, particularly if their internal connectivity is loosely structured. That's information worth reporting — it tells you which modules are "stress-relevant by eigengene correlation but diffuse internally" — and is not a defect to fix by lowering the threshold after the fact.

In [ ]:
hub_identification = {
    "method": (
        "Module-trait correlation for stress-relevant modules; "
        "kME for hub-ness scoring; fixed kME cutoff for thresholding."
    ),
    "params": {
        "stress_relevant_method":    "module_trait_correlation",
        "stress_relevant_threshold": 0.3,             # |r| >= 0.3 against stress-vs-control label
        "hubness_metric":            "kME",
        "hub_threshold_type":        "fixed_kME_cutoff",
        "hub_threshold_value":       0.8,             # kME >= 0.8
    },
    "reasoning": (
        "Module-trait correlation chosen over GO enrichment to avoid "
        "circularity (selecting modules for stress-relevance via GO "
        "would partly bake in the hub-vs-known-regulator comparison). "
        "Threshold |r| >= 0.3 follows the Langfelder & Horvath WGCNA "
        "tutorial convention for module-trait work; Hakkak 2026's "
        "tighter |r| >= 0.5 was tuned for a two-condition consensus "
        "analysis across multiple integrated datasets, while "
        "AtGenExpress's nine-condition single-dataset design has "
        "noisier per-stress correlations. kME chosen for hub-ness "
        "per the Considerations commitment (WGCNA-canonical, "
        "biologically interpretable, PyWGCNA-native). Fixed kME "
        "cutoff at 0.8 chosen over top-N as the WGCNA convention "
        "(Langfelder & Horvath 2008 framework, 0.7-0.8 the standard "
        "range); 'hub' means high centrality, not high rank within a "
        "small pool. Kumar et al. (2023) used kME-based hub "
        "identification on Arabidopsis stress data — the "
        "rationale-level precedent cited on the question page."
    ),
}

# No 'applied' field — the network and modules don't exist until
# Notebook 1; the actual hub list gets computed in Notebook 2.

print("Hub identification pre-commit recorded:")
for key, value in hub_identification["params"].items():
    print(f"  {key:>28} = {value!r}")

In [ ]:
precommit["hub_identification"] = hub_identification

print(f"`precommit` has {len(precommit)} of 6 categories filled in:")
for key in precommit:
    print(f"  - {key}")

## 6) Pre-commit your reference set, statistical framework, and background gene set

This section is the third and final pre-commit section. It covers
three categories at once because they share a single purpose: the
comparison framework Notebook 3 will use to test whether your hubs
are enriched for known stress regulators.

The three categories are:

1. **Reference set** — the list of "known stress regulators" you're
   comparing against.
2. **Statistical framework** — the test you'll use, plus
   multiple-testing correction across the two tissues.
3. **Background gene set** — the universe of genes the test runs over.

They're recorded as separate `precommit` categories because they're
substantively different decisions (different sources, different
methods, different reasoning). But they're settled together because
none of them is fully independent — the choice of reference set
constrains what's reasonable as a background, and the choice of
background constrains what's reasonable as a test.

After this section, `precommit` will be full — six of six categories.
Section 7 writes it to disk.

### 6.1) Reference set

You need a list of "known stress regulators" to compare your hubs
against. Three real candidates:

- **Hakkak 2026 Supplementary Table 3.** Roughly 60 stress-responsive
  transcription factors across 15 families, identified by Hakkak and
  Tohidfar from their multi-dataset drought and salt analysis. Already
  loaded in the project — R1-Q1 used this same table for its
  consensus comparison.

- **Wang & Singhal 2025 review list.** A larger, more recent curation
  of plant stress TFs published as a review article. Comprehensive but
  not specifically tied to *Arabidopsis* AtGenExpress-style data.

- **PlantTFDB.** The full *Arabidopsis* TF database. Largest list,
  best maintained, but covers all TFs rather than stress-specific
  regulators — so the "known stress regulator" framing breaks down.

**Working default: Hakkak 2026 Supplementary 3.** Three reasons.
First, it's stress-specific — the entries are TFs identified through
a stress-response analysis, which is what we're testing for. Second,
it's the same precedent R1-Q1 used, which keeps the rationale-level
methodology consistent across questions. Third, the file is already
in your project Drive folder (`hakkak_2026_supp3.csv`); no additional
upload step.

One small detail to record now so Notebook 3 doesn't trip on it: the
AGI column in this file is named `ORF`, not `agi`.

In [ ]:
reference_set = {
    "method": "Hakkak 2026 Supplementary Table 3 — stress-responsive TFs",
    "params": {
        "source_file": "hakkak_2026_supp3.csv",
        "agi_column": "ORF",
    },
    "reasoning": (
        "Hakkak's Supp 3 is stress-specific (TFs identified through a "
        "stress-response analysis), the same precedent R1-Q1 used for "
        "its consensus comparison, and already in the project Drive "
        "folder. Wang & Singhal's review list and PlantTFDB are the "
        "credible alternatives; Hakkak is preferred for stress "
        "specificity and rationale-level consistency."
    ),
}

# No 'applied' field — the file gets loaded and the AGI list extracted
# in Notebook 3, not here.

print("Reference set pre-commit recorded:")
for key, value in reference_set["params"].items():
    print(f"  {key:>12} = {value!r}")

### 6.2) Statistical framework

Two coupled decisions: the test, and how to correct for running it
twice (once per tissue).

**The test: hypergeometric.** For asking "is my hub list enriched for
genes in this reference set, given the background of all possible
genes," the standard tests are the hypergeometric test and Fisher's
exact test. For a 2×2 contingency table (hub-vs-not,
in-reference-vs-not), they're mathematically equivalent — the
hypergeometric is the form that ships with most numerical libraries
(`scipy.stats.hypergeom`), and it's what R1-Q1 used in
`03_consensus_compare.ipynb`. Adopting it keeps the rationale-level
pipeline consistent.

**Multiple-testing correction: Bonferroni across two tests.** You'll
run the enrichment test twice — once for shoot hubs, once for root
hubs. To keep the family-wise error rate at α = 0.05, Bonferroni
correction gives each test a per-test threshold of α = 0.025.

Bonferroni is conservative; with only two tests, the conservatism is
small (Bonferroni-corrected α = 0.025 vs an FDR-corrected α that
would be slightly larger). For two tests, Bonferroni is the simplest
defensible choice. If you decide to run more tests later (per-stress
hub enrichment, for example), revisit this in EQ#2.

In [ ]:
statistical_framework = {
    "method": "Hypergeometric enrichment test, Bonferroni correction across tissues",
    "params": {
        "test": "hypergeometric",
        "n_tests": 2,                   # one per tissue (shoot, root)
        "alpha_family_wise": 0.05,
        "alpha_per_test": 0.025,        # Bonferroni: 0.05 / 2
        "correction_method": "bonferroni",
    },
    "reasoning": (
        "Hypergeometric is the standard test for 2x2 contingency-table "
        "enrichment and is what R1-Q1 used in 03_consensus_compare.ipynb. "
        "Bonferroni correction across two tests (one per tissue) keeps "
        "family-wise alpha at 0.05 with a per-test threshold of 0.025. "
        "Conservative but defensible at this test count; revisit if more "
        "tests get added."
    ),
}

# No 'applied' field — the test gets run in Notebook 3.

print("Statistical framework pre-commit recorded:")
for key, value in statistical_framework["params"].items():
    print(f"  {key:>20} = {value!r}")

### 6.3) Background gene set

The hypergeometric test asks: of the N genes in the background, K are
in the reference set, your hub list has n genes, k of those overlap
with the reference. What's the probability of seeing k or more by
chance? The background N is what defines "chance."

Three options:

- **All 22,810 ATH1 probes.** The naive universe — every probe on the
  array. Ignores the fact that you filtered to high-MAD genes before
  building the network.

- **The MAD-filtered set per tissue (~5,700 genes).** What you
  actually fed to WGCNA. Hubs were *only ever drawn from* this set,
  so the background should match.

- **The stress-relevant modules' gene set.** Even more restricted — only genes in modules that passed the `|r| >= 0.3` stress-relevance filter. But this double-counts the stress-relevance filter on both axes of the contingency table, which biases the test.


**Working default: the MAD-filtered set per tissue.** The background
should be the set of genes that *could have been hubs* but mostly
weren't. That's the MAD-filtered set — genes that survived the
variance filter, were assigned to modules, and were eligible for hub
status. Using the full ATH1 array would let low-variance genes inflate
N artificially; using only the stress-relevant module members would
shrink the background too aggressively.

The background is per-tissue because the MAD filter ran per-tissue.
Shoot's enrichment test uses shoot's background; root's uses root's.

In [ ]:
background_gene_set = {
    "method": "MAD-filtered gene set, per tissue",
    "params": {
        "source": "Section 3 output: shoot_filtered, root_filtered",
        "per_tissue": True,
    },
    "applied": {
        "shoot": {"n_background_genes": int(shoot_filtered.shape[0])},
        "root":  {"n_background_genes": int(root_filtered.shape[0])},
    },
    "reasoning": (
        "Background should be the set of genes that could have been "
        "hubs. That's the MAD-filtered set, not the full ATH1 array "
        "(which would inflate N with low-variance genes that were "
        "never eligible) and not the stress-relevant module members "
        "(which would double-count the stress-relevance filter on "
        "both axes of the contingency table). Per-tissue because the "
        "MAD filter ran per-tissue."
    ),
}

print("Background gene set pre-commit recorded:")
for key, value in background_gene_set["params"].items():
    print(f"  {key:>15} = {value!r}")
print(f"  applied (shoot): {background_gene_set['applied']['shoot']['n_background_genes']} genes")
print(f"  applied (root):  {background_gene_set['applied']['root']['n_background_genes']} genes")

### 6.4) Append all three to `precommit`

Add the three categories to the combined dictionary. After this cell,
`precommit` is complete.

In [ ]:
precommit["reference_set"] = reference_set
precommit["statistical_framework"] = statistical_framework
precommit["background_gene_set"] = background_gene_set

print(f"`precommit` has {len(precommit)} of 6 categories filled in:")
for key in precommit:
    print(f"  - {key}")

## 7) Write `precommit.json`

`precommit` is complete with all six categories. This section
serializes it to a JSON file in your output directory —
`irilab2026_outputs/r1_q2/precommit.json`. That file is your Week 1
EQ#1 deliverable.

The save uses `indent=2` for pretty-printing so the file is readable
when you or a mentor opens it. JSON's key order is preserved as
written, so the six categories appear in the same analytical sequence
you committed to them (gene filtering → network construction → hub
identification → reference set → statistical framework → background
gene set).

In [ ]:
import json

precommit_path = OUTPUT_DIR / "precommit.json"
with open(precommit_path, "w") as f:
    json.dump(precommit, f, indent=2)

print(f"Wrote: {precommit_path}")

In [ ]:
# Confirm the file landed on disk with the expected content.
assert precommit_path.exists(), f"precommit.json was not created at {precommit_path}"

file_size = precommit_path.stat().st_size
print(f"File size: {file_size:,} bytes")
print(f"Categories serialized: {len(precommit)}")

## 8) Sanity-load

This section reads `precommit.json` back from disk and confirms the
round-trip matches what was written. Cheap insurance against typos in
the file, JSON serialization edge cases, or accidental overwrites
between Section 7 and Notebook 1 picking it up.

If the round-trip fails, the file did not save the way you think it
did, and Notebook 1 would hit the same problem when loading it. Better
to surface it now.